# GeoLifeCLEF 2025 — reproducible research pipeline

Single control notebook for matched baselines, Landsat experiments, spatial validation, multi-seed ablations, climate models and fusion. Every stage writes restartable artifacts and uses only competition data.

In [ ]:
from pathlib import Path
import json, subprocess, sys

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
DATA_ROOT = Path('/kaggle/input/competitions/geolifeclef-2025')
assert DATA_ROOT.is_dir(), 'Attach the geolifeclef-2025 competition source.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', str(ROOT)], check=True)

In [ ]:
def run_once(output, command):
    output = ROOT / output
    if output.exists():
        print(f'Reusing {output}')
        return
    subprocess.run(command, cwd=ROOT, check=True)

def read_json(path):
    return json.loads((ROOT / path).read_text(encoding='utf-8'))

## 1. Full frequency baseline

In [ ]:
run_once('artifacts/frequency_pa/metrics.json', [sys.executable, 'scripts/run_frequency_baseline.py', '--metadata-path', str(DATA_ROOT / 'GLC25_PA_metadata_train.csv'), '--report-path', 'artifacts/frequency_pa/metrics.json'])
read_json('artifacts/frequency_pa/metrics.json')

## 2. Full Landsat temporal CNN

In [ ]:
run_once('data/processed/landsat_pa_full_manifest.json', [sys.executable, 'scripts/prepare_landsat_pa.py', '--data-root', str(DATA_ROOT), '--train-output', 'data/processed/landsat_pa_full_train.npz', '--val-output', 'data/processed/landsat_pa_full_val.npz', '--manifest-path', 'data/processed/landsat_pa_full_manifest.json', '--max-train-surveys', '71190', '--max-validation-surveys', '17797'])
run_once('artifacts/landsat_tcn_full/metrics.json', [sys.executable, 'scripts/train.py', '--config', 'configs/landsat_tcn_full.yaml'])
run_once('artifacts/landsat_tcn_full/evaluation.json', [sys.executable, 'scripts/evaluate.py', '--checkpoint', 'artifacts/landsat_tcn_full/best.pt', '--split', 'data/processed/landsat_pa_full_val.npz', '--channels', '32', '--top-k', '16'])

In [ ]:
baseline = read_json('artifacts/frequency_pa/metrics.json')
landsat = read_json('artifacts/landsat_tcn_full/evaluation.json')
manifest = read_json('data/processed/landsat_pa_full_manifest.json')
summary = {'train_surveys': manifest['train_samples'], 'validation_surveys': manifest['validation_samples'], 'frequency_sample_f1': baseline.get('sample_f1_top_k'), 'landsat_top16_sample_f1': landsat.get('sample_f1_top_k'), 'official_private_leaderboard_target': 0.2302}
summary

## 3. Spatial holdout audit

Choose the geographic holdout with a pre-registered rule, before inspecting model performance. This prevents selecting an easy region after the fact.

In [ ]:
run_once('data/reports/spatial_split_audit.json', [sys.executable, 'scripts/audit_spatial_split.py', '--metadata-path', str(DATA_ROOT / 'GLC25_PA_metadata_train.csv'), '--report-path', 'data/reports/spatial_split_audit.json'])
spatial_audit = read_json('data/reports/spatial_split_audit.json')
{'countries': spatial_audit['country_count'], 'regions': spatial_audit['region_count'], 'recommended_holdout': spatial_audit['recommended_holdout']}

## 4. SOTA-oriented multimodal spatial comparison

Both models use identical frozen split hashes. Prediction policy is fitted only on non-Netherlands calibration blocks; Netherlands remains untouched until the final comparison.

In [ ]:
MODE = 'smoke'  # change to 'full' only after the smoke comparison passes
data_dir = f'data/processed/sota_spatial_{MODE}'
artifact_dir = f'artifacts/sota_spatial_{MODE}'
limits = ['--max-train-surveys', '12000', '--max-calibration-surveys', '2000', '--max-validation-surveys', '3000'] if MODE == 'smoke' else []
run_once(f'{data_dir}/manifest.json', [sys.executable, 'scripts/prepare_spatial_multimodal.py', '--data-root', str(DATA_ROOT), '--output-dir', data_dir, '--holdout-country', 'Netherlands', '--image-size', '32', *limits])
training_args = ['--batch-size', '96', '--reference-epochs', '3', '--fusion-epochs', '4', '--model-dim', '96', '--max-hours', '3.0'] if MODE == 'smoke' else ['--batch-size', '64', '--reference-epochs', '8', '--fusion-epochs', '10', '--model-dim', '192', '--max-hours', '10.5']
run_once(f'{artifact_dir}/comparison.json', [sys.executable, 'scripts/train_spatial_competition.py', '--data-dir', data_dir, '--output-dir', artifact_dir, *training_args])
read_json(f'{artifact_dir}/comparison.json')

## 5. Promotion rule

Promote to the full run only if tests pass, split hashes match, the fusion candidate beats the reference on Netherlands sample-F1, and measured runtime projects below 10.5 hours. A hidden-test score above 0.2302 is still required for an actual SOTA claim.